<a href="https://colab.research.google.com/github/otaviau/PCVK_Genap_2026/blob/main/Week7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 7 – Segmentasi Citra dengan Thresholding
Jawaban Jobsheet Modul P7 (Pengolahan Citra dan Visi Komputer).

## Persiapan (Soal 1 & 2)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#Library yang digunakan
import cv2 as cv
import numpy as np
import math
import pandas as pd
import matplotlib.pyplot as plt
from collections import deque
from scipy import ndimage as ndi
from skimage import data, filters, color, segmentation, feature, morphology
from skimage.filters import gaussian
from skimage.segmentation import flood

# Folder tempat gambar disimpan di Drive
BASE = '/content/drive/MyDrive/PCVK/Images/'

## Soal 3 – Global Threshold manual (T = 200)

Idenya sederhana: bandingkan setiap piksel dengan `T`, lalu ubah nilainya sesuai aturan tabel.
Kita pakai `np.where(kondisi, nilai_jika_benar, nilai_jika_salah)` agar tidak perlu loop.

In [ ]:
# Gambar gradasi hitam -> putih (256 baris x 512 kolom)
gradient = np.tile(np.linspace(0, 255, 512), (256, 1)).astype(np.uint8)

def global_threshold(img, T, maxval, jenis):
    if jenis == 'BINARY':
        hasil = np.where(img > T, maxval, 0)
    elif jenis == 'BINARY_INV':
        hasil = np.where(img > T, 0, maxval)
    elif jenis == 'TRUNC':
        hasil = np.where(img > T, T, img)
    elif jenis == 'TOZERO':
        hasil = np.where(img > T, img, 0)
    elif jenis == 'TOZERO_INV':
        hasil = np.where(img > T, 0, img)
    return hasil.astype(np.uint8)

T = 200
maxval = 255
jenis_list = ['BINARY', 'BINARY_INV', 'TRUNC', 'TOZERO', 'TOZERO_INV']

titles = ['Original Image'] + jenis_list
images = [gradient] + [global_threshold(gradient, T, maxval, j) for j in jenis_list]

plt.figure(figsize=(15, 5))
for i in range(len(images)):
    plt.subplot(2, 3, i+1)
    plt.imshow(images[i], 'gray', vmin=0, vmax=255, interpolation='nearest')
    plt.title(titles[i])
    plt.xticks([]), plt.yticks([])
plt.show()

## Soal 4 – Adaptive Threshold manual (Mean & Gaussian)

Langkah per piksel: ambil jendela 11×11 → hitung rata-rata (atau rata-rata berbobot Gaussian) → kurangi `C=2` → bandingkan.
Tanpa `cv.threshold`, `cv.adaptiveThreshold`, ataupun filter siap pakai. Kita geser gambar 121 kali (sesuai isi jendela) lalu jumlahkan dengan bobotnya — hasilnya sama dengan rata-rata jendela.

In [ ]:
gray_sudoku = cv.imread(BASE + 'sudoku-original.jpg', 0)   # 0 = langsung grayscale

def adaptive_manual(gray, ukuran=11, C=2, metode='mean', sigma=2):
    r = ukuran // 2
    g = gray.astype(np.float32)
    H, W = g.shape
    p = np.pad(g, r, mode='edge')            # tambah tepi agar jendela tidak keluar gambar

    # Kernel (bobot jendela)
    if metode == 'mean':
        kernel = np.ones((ukuran, ukuran)) / (ukuran * ukuran)
    else:  # gaussian
        ax = np.arange(-r, r + 1)
        k1 = np.exp(-(ax ** 2) / (2 * sigma ** 2))
        kernel = np.outer(k1, k1)
        kernel = kernel / kernel.sum()       # jumlah bobot = 1

    # Rata-rata lokal = jumlah (bobot x gambar yang digeser)
    rata = np.zeros_like(g)
    for i in range(ukuran):
        for j in range(ukuran):
            rata += kernel[i, j] * p[i:i+H, j:j+W]

    T_lokal = rata - C
    return np.where(g > T_lokal, 255, 0).astype(np.uint8)

# Global threshold manual (T = 127)
global_127 = np.where(gray_sudoku > 127, 255, 0).astype(np.uint8)
adp_mean = adaptive_manual(gray_sudoku, 11, 2, 'mean')
adp_gauss = adaptive_manual(gray_sudoku, 11, 2, 'gaussian', sigma=2)

titles = ['Citra Asli', 'Global Thresholding (T = 127)', 'Adaptive Mean — Manual', 'Adaptive Gaussian — Manual']
hasil = [gray_sudoku, global_127, adp_mean, adp_gauss]
plt.figure(figsize=(16, 5))
for i in range(4):
    plt.subplot(1, 4, i+1)
    plt.imshow(hasil[i], 'gray', vmin=0, vmax=255)
    plt.title(titles[i])
    plt.xticks([]), plt.yticks([])
plt.show()

## Soal 5A – Otsu manual (tanpa library) pada Lena low contrast

Otsu mencoba semua `T` dari 0–254, lalu memilih `T` dengan **between-class variance terbesar**:
`σ²_B = w_bg × w_fg × (mean_bg − mean_fg)²`.

In [ ]:
def otsu_manual(gray):
    hist = np.bincount(gray.ravel(), minlength=256).astype(np.float64)
    p = hist / hist.sum()                    # probabilitas tiap level
    level = np.arange(256)

    T_terbaik, var_terbaik = 0, -1
    for T in range(255):
        w_bg = p[:T+1].sum()                 # background: intensitas <= T
        w_fg = p[T+1:].sum()                 # foreground: intensitas  > T
        if w_bg == 0 or w_fg == 0:
            continue
        mu_bg = (level[:T+1] * p[:T+1]).sum() / w_bg
        mu_fg = (level[T+1:] * p[T+1:]).sum() / w_fg
        var_b = w_bg * w_fg * (mu_bg - mu_fg) ** 2
        if var_b > var_terbaik:
            T_terbaik, var_terbaik = T, var_b

    hasil = np.where(gray > T_terbaik, 255, 0).astype(np.uint8)
    return hasil, T_terbaik

img = cv.imread(BASE + 'lena_gs_lc2.jpg', 0)
blur = cv.GaussianBlur(img, (5, 5), 0)
thresh = 127

otsu_biner, otsu_thresh = otsu_manual(blur)
ret_lib, _ = cv.threshold(blur, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
print('Threshold Otsu manual :', otsu_thresh)
print('Threshold Otsu OpenCV :', ret_lib, '(harus sama / selisih paling 1)')

th1 = np.where(blur > thresh, 255, 0).astype(np.uint8)
titles = ['Citra Asli', 'Global Thresholding (Thresh = 127)', "Otsu's Thresholding (v = %d)" % otsu_thresh]
citra3 = [blur, th1, otsu_biner]
plt.figure(figsize=(15, 5))
for i in range(3):
    plt.subplot(1, 3, i+1)
    plt.imshow(citra3[i], 'gray', vmin=0, vmax=255)
    plt.title(titles[i])
    plt.xticks([]), plt.yticks([])
plt.show()

### Jawaban Pertanyaan Soal 5A — lihat histogramnya

In [ ]:
print('Nilai piksel terkecil :', blur.min())
print('Nilai piksel terbesar :', blur.max())

plt.figure(figsize=(8, 4))
plt.hist(blur.ravel(), 256, [0, 256], color='steelblue')
plt.axvline(127, color='red', label='Global T = 127')
plt.axvline(otsu_thresh, color='black', label='Otsu T = %d' % otsu_thresh)
plt.axvspan(blur.min(), blur.max(), color='yellow', alpha=0.2, label='rentang piksel Lena')
plt.xlabel('Intensitas'); plt.ylabel('Jumlah piksel')
plt.title('Histogram Lena low contrast'); plt.legend()
plt.show()

# Coba ambang yang masih "masuk" di rentang piksel Lena
T_coba = [90, 95, 100]
plt.figure(figsize=(12, 4))
for i, t in enumerate(T_coba):
    plt.subplot(1, 3, i+1)
    plt.imshow(np.where(blur > t, 255, 0), 'gray')
    plt.title('Thresh = %d' % t); plt.xticks([]), plt.yticks([])
plt.show()

**Jawaban:**

- **Mengapa hitam semua?** Seluruh piksel Lena low contrast hanya bernilai sekitar **85–108** (histogram berupa satu "gunung" sempit di sisi kiri). Garis ambang **127** ada di sebelah **kanan** semua piksel, sehingga untuk setiap piksel berlaku `piksel ≤ 127` → diubah menjadi **0 (hitam)**. Tidak ada piksel yang melewati ambang, jadi tidak ada yang menjadi putih.
- **Ambang agar Lena masih terlihat:** ambang harus berada **di dalam rentang piksel (85–108)**, misalnya sekitar **90–100** (Otsu memilih ± 92). Di bawah 85 gambar putih semua, di atas 108 gambar hitam semua.

## Soal 5B – Otsu pada KTP sendiri
Ganti nama file `ktp_fulan.jpg` dengan nama file KTP kamu bila berbeda.

In [ ]:
ktp_gray = cv.imread(BASE + 'ktp_fulan.jpg', 0)
ktp_blur = cv.GaussianBlur(ktp_gray, (5, 5), 0)

ktp_otsu, ktp_T = otsu_manual(ktp_blur)
ktp_global = np.where(ktp_blur > 127, 255, 0).astype(np.uint8)

titles = ['Citra Asli', 'Global Thresholding (Thresh = 127)', "Otsu's Thresholding (v = %d)" % ktp_T]
hasil = [ktp_gray, ktp_global, ktp_otsu]
plt.figure(figsize=(18, 5))
for i in range(3):
    plt.subplot(1, 3, i+1)
    plt.imshow(hasil[i], 'gray', vmin=0, vmax=255)
    plt.title(titles[i]); plt.xticks([]), plt.yticks([])
plt.show()

## Soal 6A – Region Growing manual pada `data.coins()`

Cara kerja: mulai dari **seed**, lalu tambahkan tetangga (4 arah) yang selisih intensitasnya dengan seed ≤ `T`. Ulangi sampai tidak ada tetangga yang cocok.
Koordinat seed ditulis `(baris, kolom)`.

In [ ]:
gray = data.coins()          # grayscale, 303 x 384

def region_growing(gray, seed, T):
    H, W = gray.shape
    nilai_seed = int(gray[seed])
    mask = np.zeros((H, W), dtype=bool)
    mask[seed] = True
    antrian = deque([seed])

    while antrian:
        y, x = antrian.popleft()
        for dy, dx in ((-1, 0), (1, 0), (0, -1), (0, 1)):     # 4 tetangga
            ny, nx = y + dy, x + dx
            if 0 <= ny < H and 0 <= nx < W and not mask[ny, nx]:
                if abs(int(gray[ny, nx]) - nilai_seed) <= T:
                    mask[ny, nx] = True
                    antrian.append((ny, nx))
    return mask

seeds = {'Seed di dalam koin': (55, 45),
         'Seed di latar':      (150, 150),
         'Seed di tepi koin':  (55, 22)}
T_list = [5, 20, 60]

def tampilkan_grid(fungsi, judul):
    fig, axes = plt.subplots(3, 3, figsize=(14, 11))
    for r, (nama, seed) in enumerate(seeds.items()):
        for c, T in enumerate(T_list):
            mask = fungsi(gray, seed, T)
            rgb = np.stack([gray] * 3, axis=-1).copy()
            rgb[mask] = [255, 0, 0]                    # region = merah
            ax = axes[r, c]
            ax.imshow(rgb)
            ax.plot(seed[1], seed[0], 'c+', markersize=14, markeredgewidth=2)   # tanda seed
            ax.set_title('%s\nT = %d (%d piksel)' % (nama, T, mask.sum()), fontsize=9)
            ax.axis('off')
    fig.suptitle(judul)
    plt.tight_layout()
    plt.show()

for nama, seed in seeds.items():
    print(nama, seed, '-> intensitas', gray[seed])

tampilkan_grid(region_growing, 'Region Growing manual (baris = seed, kolom = T)')

## Soal 6B – Pakai `skimage.segmentation.flood()`

In [ ]:
def region_flood(gray, seed, T):
    return flood(gray, seed_point=seed, tolerance=T, connectivity=1)   # 1 = 4 tetangga

tampilkan_grid(region_flood, 'skimage flood() (baris = seed, kolom = T)')

# Bandingkan dengan versi manual
for nama, seed in seeds.items():
    for T in T_list:
        sama = np.array_equal(region_growing(gray, seed, T), region_flood(gray, seed, T))
        print('%-20s T=%-3d hasil manual == flood ? %s' % (nama, T, sama))

**Catatan hasil yang biasanya terlihat** (angka piksel bisa sedikit berbeda di komputermu):
- **T = 5** → region sangat kecil, hanya bercak di sekitar seed (tekstur koin membuat nilai piksel berubah-ubah).
- **T = 20** → region masih terbatas; seed di latar sudah menyebar luas di area gelap.
- **T = 60** → region terlalu longgar: seed di latar menutupi hampir seluruh gambar, seed di koin/tepi mulai melebar ke bagian koin yang lebih luas.
- Seed di **tepi koin** paling tidak stabil, karena nilainya berada di antara koin dan latar (T = 5 hanya menghasilkan region sangat kecil).

## Soal 7 – Segmentasi warna: tampilkan hanya warna biru

Langkah: (1) K-Means (K=8) untuk menyederhanakan warna → (2) ubah ke HSV → (3) ambil piksel yang masuk rentang biru → sisanya dihitamkan.

In [ ]:
from sklearn.cluster import KMeans

ktp = cv.imread(BASE + 'ktp_fulan.jpg')
ktp = cv.cvtColor(ktp, cv.COLOR_BGR2RGB)

# Perkecil agar K-Means cepat
lebar = 800
tinggi = int(ktp.shape[0] * lebar / ktp.shape[1])
ktp = cv.resize(ktp, (lebar, tinggi))

# 1. K-Means (K = 8)
Z = np.float32(ktp.reshape((-1, 3)))
kmeans = KMeans(n_clusters=8, random_state=42, n_init=10)
kmeans.fit(Z)
pusat = np.uint8(kmeans.cluster_centers_)
ktp_kmeans = pusat[kmeans.labels_].reshape(ktp.shape)

# 2. Ubah ke HSV (di OpenCV: H 0-179, S 0-255, V 0-255)
hsv = cv.cvtColor(ktp_kmeans, cv.COLOR_RGB2HSV)
H, S, V = hsv[:, :, 0], hsv[:, :, 1], hsv[:, :, 2]

# 3. Rentang biru dari soal
kelompok_biru = (H >= 85) & (H <= 130) & (S >= 25) & (V >= 40)

hanya_biru = np.zeros_like(ktp_kmeans)           # semua hitam
hanya_biru[kelompok_biru] = ktp_kmeans[kelompok_biru]

titles = ['Citra Asli', 'Segmentasi K-Means (K=8)', 'Hanya Warna Biru']
hasil = [ktp, ktp_kmeans, hanya_biru]
plt.figure(figsize=(18, 5))
for i in range(3):
    plt.subplot(1, 3, i+1)
    plt.imshow(hasil[i]); plt.title(titles[i]); plt.axis('off')
plt.show()

---
# D2 – Tugas Kelompok: OCR pada KTP
Instal Tesseract + model bahasa Indonesia (jalankan sekali, tunggu sampai selesai).

In [ ]:
!apt-get -qq update
!apt-get -qq install tesseract-ocr tesseract-ocr-ind
!pip -q install pytesseract

### Langkah 1 – Siapkan tiga citra (ukuran sama)
Satu citra KTP milik satu anggota. Otsu dan Adaptive dikerjakan pada versi **grayscale**.

In [ ]:
import re, difflib
import pytesseract

# Ganti dengan KTP anggota kelompok
FILE_KTP = BASE + 'ktp_fulan.jpg'

bgr = cv.imread(FILE_KTP)
lebar = 1000
tinggi = int(bgr.shape[0] * lebar / bgr.shape[1])
bgr = cv.resize(bgr, (lebar, tinggi))                 # ukuran sama untuk ketiganya

img_warna = cv.cvtColor(bgr, cv.COLOR_BGR2RGB)        # citra berwarna asli
gray = cv.cvtColor(bgr, cv.COLOR_BGR2GRAY)            # grayscale dulu

_, img_otsu = cv.threshold(gray, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
img_adaptive = cv.adaptiveThreshold(gray, 255, cv.ADAPTIVE_THRESH_GAUSSIAN_C,
                                    cv.THRESH_BINARY, 31, 10)

citra_ocr = {'Berwarna': img_warna, 'Otsu': img_otsu, 'Adaptive': img_adaptive}

plt.figure(figsize=(18, 5))
for i, (nama, im) in enumerate(citra_ocr.items()):
    plt.subplot(1, 3, i+1)
    plt.imshow(im, 'gray' if im.ndim == 2 else None); plt.title(nama); plt.axis('off')
plt.show()

### Langkah 2 & 3 – Baca teks (bahasa & `--psm` sama) dan tampilkan kotak hijau (confidence > 60)

In [ ]:
BAHASA = 'ind'
CONFIG = '--psm 6'
BATAS_CONF = 60

hasil_teks, hasil_data = {}, {}
for nama, im in citra_ocr.items():
    hasil_teks[nama] = pytesseract.image_to_string(im, lang=BAHASA, config=CONFIG)
    hasil_data[nama] = pytesseract.image_to_data(im, lang=BAHASA, config=CONFIG,
                                                 output_type=pytesseract.Output.DICT)

def gambar_kotak(im, d, batas=BATAS_CONF):
    out = cv.cvtColor(im, cv.COLOR_GRAY2RGB) if im.ndim == 2 else im.copy()
    for i in range(len(d['text'])):
        if d['text'][i].strip() != '' and float(d['conf'][i]) > batas:
            x, y, w, h = d['left'][i], d['top'][i], d['width'][i], d['height'][i]
            cv.rectangle(out, (x, y), (x + w, y + h), (0, 255, 0), 2)
    return out

plt.figure(figsize=(18, 5))
for i, nama in enumerate(citra_ocr):
    plt.subplot(1, 3, i+1)
    plt.imshow(gambar_kotak(citra_ocr[nama], hasil_data[nama])); plt.title('Area Teks — ' + nama); plt.axis('off')
plt.show()

for nama in citra_ocr:
    print('========== %s ==========' % nama.upper())
    print(hasil_teks[nama])

### Langkah 4 – Tabel perbandingan 10 item
**Edit daftar `acuan` di bawah** dengan 10 kata/angka yang terlihat jelas di KTP kamu (tulis persis seperti di gambar).
Contoh bawaan di bawah memakai KTP contoh dari jobsheet.

In [ ]:
acuan = ['PROVINSI', 'TIMUR', 'SURABAYA', '01-01-1990', 'LAKI-LAKI',
         'ISLAM', 'KAWIN', 'KARYAWAN', 'DUREN', 'SEUMUR']      # <-- GANTI sesuai KTP kamu

def normal(s):
    return re.sub(r'\s+', '', s.lower()).strip(':,;.')        # huruf kecil, tanpa spasi

def token_ocr(teks):
    return [normal(t) for t in teks.split() if normal(t) != '']

def bacaan_terdekat(item, teks):
    tokens = token_ocr(teks)
    cocok = difflib.get_close_matches(normal(item), tokens, n=1, cutoff=0.5)
    return cocok[0].upper() if cocok else '(tidak terbaca)'

def rata_confidence(d):
    nilai = [float(c) for t, c in zip(d['text'], d['conf']) if t.strip() != '' and float(c) >= 0]
    return round(float(np.mean(nilai)), 1) if nilai else 0.0

tabel = pd.DataFrame({'No': range(1, len(acuan) + 1), 'Teks sebenarnya': acuan})
tabel['OCR citra berwarna'] = [bacaan_terdekat(a, hasil_teks['Berwarna']) for a in acuan]
tabel['OCR Otsu']           = [bacaan_terdekat(a, hasil_teks['Otsu']) for a in acuan]
tabel['OCR Adaptive']       = [bacaan_terdekat(a, hasil_teks['Adaptive']) for a in acuan]
tabel

### Langkah 5 – Persentase kecocokan

In [ ]:
baris = []
for nama, label in [('Berwarna', 'Berwarna asli'), ('Otsu', 'Segmentasi Otsu'), ('Adaptive', 'Segmentasi Adaptive')]:
    tokens = token_ocr(hasil_teks[nama])
    tepat = sum(1 for a in acuan if normal(a) in tokens)       # cocok persis (abaikan besar/kecil & spasi)
    baris.append({'Citra masukan OCR': label,
                  'Item terbaca tepat': '%d dari %d' % (tepat, len(acuan)),
                  'Kecocokan (%)': tepat / len(acuan) * 100,
                  'Rata-rata confidence': rata_confidence(hasil_data[nama])})
ringkasan = pd.DataFrame(baris)
ringkasan